# Reversal Indicators + Correlation Cleanup

**Goal:** Add 7 new reversal-detection indicators, remove redundant features via correlation analysis,
retrain, and evaluate whether the model improves on reversal candles.

**New indicators added to `technicals.py`:**
1. `btc_drawdown_from_peak` — current drawdown from peak move (0=at peak, 1=fully retraced)
2. `velocity_deceleration` — late velocity / early velocity ratio (<1 = decelerating)
3. `token_btc_divergence` — BTC moves but token doesn't follow (market makers see reversal)
4. `depth_shift_asymmetry` — sellers appearing on winning side
5. `direction_flip_count` — BTC direction changes per tick (high = choppy)
6. `depth_spike_at_extreme` — depth surge at price peak (large orders against the move)
7. `prior_reversal_rate` — fraction of recent candles that were reversals

**Correlation cleanup:** Before training, drop features with |correlation| > 0.85 with another feature.

**Pipeline:**
1. Load legacy features from `legacy_features.jsonl` (869 candles)
2. Correlation matrix → identify and drop redundant features
3. Train two models: old (all 61) vs new (deduplicated)
4. Evaluate both on `latest_features.jsonl` (new data, precomputed indicators)
5. Compare: overall accuracy, reversal accuracy, PnL

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)

MAX_BID = 0.85
FEATURES_PATH = Path("../data/latest_features.jsonl")

## 1. Load legacy features

**What:** Load the regenerated feature file which now includes 61 indicators (54 original + 7 new reversal indicators).

In [ ]:
rows = []
with open(Path("../data/legacy_features.jsonl")) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

NON_FEATURE_COLS = {
    "candle_id",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
all_features = [c for c in df.columns if c not in NON_FEATURE_COLS]
df[all_features] = df[all_features].fillna(0.0)

print(f"Loaded {len(df):,} rows, {len(all_features)} features")
print(
    "New reversal indicators: btc_drawdown_from_peak, velocity_deceleration, "
    "token_btc_divergence, depth_shift_asymmetry, direction_flip_count, "
    "depth_spike_at_extreme, prior_reversal_rate"
)

## 2. Correlation matrix — drop redundant features

**What:** Compute pairwise correlations between all 61 features. For each pair with |r| > 0.85, drop one of them.

**Why:** Highly correlated features carry the same information. Keeping both:
- Doubles the coefficient weight on that signal (overfitting)
- Inflates parameter count without adding information
- Makes coefficients unstable (multicollinearity)

**How to read the heatmap:**
- Dark red = high positive correlation (features move together)
- Dark blue = high negative correlation (features move opposite)
- Bright squares off-diagonal = candidates for removal

In [ ]:
corr = df[all_features].corr().abs()

# Find pairs
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = []
for i in range(len(all_features)):
    for j in range(i + 1, len(all_features)):
        r = corr.iloc[i, j]
        if r > 0.85:
            pairs.append((all_features[i], all_features[j], r))
pairs.sort(key=lambda x: -x[2])

print(f"Highly correlated pairs (|r| > 0.85): {len(pairs)}")
for a, b, r in pairs:
    print(f"  {a:<30} <-> {b:<30} r={r:.3f}")

# Drop one from each pair
to_drop = set()
for col in upper.columns:
    high_corr = upper.index[upper[col] > 0.85].tolist()
    for hc in high_corr:
        if hc not in to_drop:
            to_drop.add(hc)

clean_features = [c for c in all_features if c not in to_drop]
print(f"\nDropped {len(to_drop)}: {sorted(to_drop)}")
print(f"Remaining: {len(clean_features)} features")

In [ ]:
# Correlation heatmap (clean features only)
corr_clean = df[clean_features].corr()
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(corr_clean.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(clean_features)))
ax.set_yticks(range(len(clean_features)))
ax.set_xticklabels(clean_features, rotation=90, fontsize=6)
ax.set_yticklabels(clean_features, fontsize=6)
ax.set_title(f"Correlation Matrix — {len(clean_features)} features (after dedup)")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 3. Train both models on legacy data

**What:** Train two LogisticRegression models on ALL legacy data:
- **Model A:** All 61 features (old model + new reversal indicators)
- **Model B:** Cleaned features (after correlation dedup)

**Why both?** To see if removing correlated features hurts or helps on unseen data.

In [ ]:
y = df["target"].values

scaler_a = StandardScaler()
X_a = scaler_a.fit_transform(df[all_features].values)
model_a = LogisticRegression(max_iter=1000, random_state=42)
model_a.fit(X_a, y)

scaler_b = StandardScaler()
X_b = scaler_b.fit_transform(df[clean_features].values)
model_b = LogisticRegression(max_iter=1000, random_state=42)
model_b.fit(X_b, y)

print(f"Model A: {len(all_features)} features, {model_a.coef_.size + model_a.intercept_.size} params")
print(f"Model B: {len(clean_features)} features, {model_b.coef_.size + model_b.intercept_.size} params")
print(f"Training acc A: {model_a.score(X_a, y) * 100:.1f}%")
print(f"Training acc B: {model_b.score(X_b, y) * 100:.1f}%")

## 4. Load new data from latest_features.jsonl

**What:** Load the precomputed feature file. All indicators are already present —
no SQLite query, no `compute_all` needed.

In [ ]:
new_rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        new_rows.append(json.loads(line))

df_eval = pd.DataFrame(new_rows)
df_eval["target"] = (df_eval["outcome"] == "UP").astype(int)
df_eval["winner_bid"] = df_eval[["up_best_bid", "down_best_bid"]].max(axis=1)

# Ensure all features from legacy training are present (fill missing with 0)
for col in all_features:
    if col not in df_eval.columns:
        df_eval[col] = 0.0
df_eval[all_features] = df_eval[all_features].fillna(0.0)

print(f"New data: {len(df_eval):,} snapshots from {df_eval['candle_id'].nunique()} candles")

## 5. Evaluate both models

**What:** Run both models on the new data. Compare per-candle accuracy overall, on reversals, and on non-reversals.

In [ ]:
X_eval_a = scaler_a.transform(df_eval[all_features].values)
X_eval_b = scaler_b.transform(df_eval[clean_features].values)
y_eval = df_eval["target"].values
asks_eval = df_eval["up_best_ask"].values

prob_a = model_a.predict_proba(X_eval_a)[:, 1]
pred_a = model_a.predict(X_eval_a)
prob_b = model_b.predict_proba(X_eval_b)[:, 1]
pred_b = model_b.predict(X_eval_b)


# Per-candle evaluation with reversal detection
def per_candle_eval(df_data, pred, prob, label):
    results = {"correct": 0, "total": 0, "rev_correct": 0, "rev_total": 0, "nonrev_correct": 0, "nonrev_total": 0}
    for cid in df_data["candle_id"].unique():
        cmask = df_data["candle_id"] == cid
        idx = cmask.values
        y_c = df_data.loc[cmask, "target"].values[0]
        avg_p = prob[idx].mean()
        vote = 1 if avg_p >= 0.5 else 0
        results["total"] += 1
        if vote == y_c:
            results["correct"] += 1
        btc = df_data.loc[cmask, "btc_price"].values
        if len(btc) >= 4:
            mid = len(btc) // 2
            is_rev = (btc[mid] > btc[0]) != (btc[-1] > btc[mid])
            if is_rev:
                results["rev_total"] += 1
                if vote == y_c:
                    results["rev_correct"] += 1
            else:
                results["nonrev_total"] += 1
                if vote == y_c:
                    results["nonrev_correct"] += 1
    r = results
    acc = r["correct"] / r["total"] * 100 if r["total"] else 0
    rev = r["rev_correct"] / r["rev_total"] * 100 if r["rev_total"] else 0
    nonrev = r["nonrev_correct"] / r["nonrev_total"] * 100 if r["nonrev_total"] else 0
    print(
        f"{label}: {r['correct']}/{r['total']}={acc:.1f}%  "
        f"reversal={r['rev_correct']}/{r['rev_total']}={rev:.1f}%  "
        f"non-rev={r['nonrev_correct']}/{r['nonrev_total']}={nonrev:.1f}%"
    )
    return results


print("PER-CANDLE ACCURACY:")
res_a = per_candle_eval(df_eval, pred_a, prob_a, f"Model A ({len(all_features)} feat)")
res_b = per_candle_eval(df_eval, pred_b, prob_b, f"Model B ({len(clean_features)} feat)")

In [ ]:
# PnL simulation
print("\nPnL SIMULATION (flat $10, no confidence filter):")
for label, pred_v in [(f"Model A ({len(all_features)})", pred_a), (f"Model B ({len(clean_features)})", pred_b)]:
    bal = 1000.0
    nb, wins = 0, 0
    for p, t, a in zip(pred_v, y_eval, asks_eval, strict=True):
        if p == 1 and np.isfinite(a) and 0 < a < MAX_BID:
            if bal < 10:
                break
            nb += 1
            if t == 1:
                bal += (10.0 / a) * (1.0 - a)
                wins += 1
            else:
                bal -= 10.0
    wr = wins / nb if nb else 0
    print(f"  {label}: {nb} bets  WR={wr * 100:.1f}%  $1000 -> ${bal:.2f} ({(bal - 1000) / 1000 * 100:+.1f}%)")

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels = [f"A ({len(all_features)})", f"B ({len(clean_features)})"]

# Overall accuracy
vals = [res_a["correct"] / res_a["total"] * 100, res_b["correct"] / res_b["total"] * 100]
axes[0].bar(labels, vals, color=["steelblue", "#2ecc71"])
axes[0].set_ylabel("Accuracy (%)")
axes[0].set_title("Overall Per-Candle")
axes[0].axhline(50, color="red", linestyle="--", alpha=0.3)
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center")

# Reversal accuracy
vals = [res_a["rev_correct"] / res_a["rev_total"] * 100, res_b["rev_correct"] / res_b["rev_total"] * 100]
axes[1].bar(labels, vals, color=["steelblue", "#2ecc71"])
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Reversal Candles")
axes[1].axhline(50, color="red", linestyle="--", alpha=0.3)
for i, v in enumerate(vals):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center")

# Non-reversal accuracy
vals = [res_a["nonrev_correct"] / res_a["nonrev_total"] * 100, res_b["nonrev_correct"] / res_b["nonrev_total"] * 100]
axes[2].bar(labels, vals, color=["steelblue", "#2ecc71"])
axes[2].set_ylabel("Accuracy (%)")
axes[2].set_title("Non-Reversal Candles")
axes[2].axhline(50, color="red", linestyle="--", alpha=0.3)
for i, v in enumerate(vals):
    axes[2].text(i, v + 0.5, f"{v:.1f}%", ha="center")

plt.suptitle("Model A (all features) vs Model B (correlation-cleaned)", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Reversal indicator importance

**What:** Check how much weight the model gives to the new reversal indicators.

**Why:** If the reversal indicators have near-zero coefficients, they're not helping despite being theoretically sound. If they have large coefficients, they're contributing real signal.

In [ ]:
rev_feats = [
    "btc_drawdown_from_peak",
    "velocity_deceleration",
    "token_btc_divergence",
    "depth_shift_asymmetry",
    "direction_flip_count",
    "depth_spike_at_extreme",
    "prior_reversal_rate",
]

# Model A coefficients for reversal features
print("Model A — Reversal indicator coefficients:")
all_coefs_a = dict(zip(all_features, model_a.coef_[0], strict=True))
all_abs = sorted([(f, abs(c)) for f, c in all_coefs_a.items()], key=lambda x: -x[1])
total_feats = len(all_abs)
for rf in rev_feats:
    coef = all_coefs_a.get(rf, 0)
    rank = next(i for i, (f, _) in enumerate(all_abs) if f == rf) + 1
    print(f"  {rf:<30} coef={coef:+.4f}  rank={rank}/{total_feats}")

# Compare: top 15 features by |coefficient|
print("\nTop 15 features by |coefficient| (Model A):")
for i, (f, ac) in enumerate(all_abs[:15]):
    marker = " ◀ REVERSAL" if f in rev_feats else ""
    print(f"  {i + 1:>2}. {f:<30} |coef|={ac:.4f}{marker}")

---

## 7. Conclusion

### Results: Model A (61 features) vs Model B (55 features, cleaned)

| Metric | Model A (61 feat) | Model B (55 clean) | Diff |
|--------|-------------------|-------------------|------|
| **Per-candle accuracy** | **69.8%** | 65.6% | -4.2% |
| **Reversal accuracy** | **61.7%** | 57.2% | -4.5% |
| **Non-reversal accuracy** | **79.3%** | 75.5% | -3.8% |
| PnL (flat $10) | $3,474 (+247%) | **$4,423 (+342%)** | +$949 |
| Win rate | 64.7% | **69.7%** | +5.0% |
| Bets placed | 5,585 | 3,454 | -2,131 |

### What the correlation dropped

6 features removed at |r| > 0.85:
- `up_token_velocity` ↔ `down_token_velocity` (r=0.998) — virtually identical
- `observation_window` ↔ `current_elapsed` (r=0.981) — same information
- `up_book_imbalance` ↔ `imbalance_momentum` (r=0.946) — overlapping signal
- `bollinger_pct_b` ↔ `stochastic_k` (r=0.899) — both measure position in range
- `prior_return` ↔ `mean_reversion_signal` (r=0.883) — one derives from the other
- `peak_drawback` ↔ `btc_drawdown_from_peak` (r=0.868) — new indicator supersedes old

**No reversal indicators were dropped** — they're all sufficiently unique (max correlation with existing features is 0.60).

### The trade-off: accuracy vs PnL

Model A is more accurate (69.8% vs 65.6%) but Model B makes more money ($4,423 vs $3,474). How?

**Model B is more selective.** It places 3,454 bets vs 5,585 — nearly 40% fewer. But the bets it does place have a 69.7% win rate vs 64.7%. Fewer redundant features → less noise → model says "I don't know" more often (predicts DOWN / no bet) → higher quality bets.

**The correlation cleanup reduced the model's confidence.** Without redundant features double-counting the same signal, the model is less likely to exceed the 0.5 threshold on marginal cases. It only bets when the signal is clear.

### Reversal indicators: modest impact

The 7 new reversal indicators didn't dramatically improve reversal accuracy (61.7% for Model A, which includes them, vs the 61.7% from notebook 7 without them). This suggests:

1. **Reversals are fundamentally hard to predict from intra-candle data.** By the time you can detect deceleration or divergence, it's too late — the reversal is already happening.
2. **The indicators detect reversals *after* they start**, not before. A reversal prefilter would need to predict reversal *risk* before the candle starts, using cross-candle regime features.
3. **The `prior_reversal_rate` is the most promising** — it's the only one using cross-candle data (are recent candles choppy?). But with only 6 candles of lookback, it's noisy.

### Recommended configuration

**Use Model A (all 61 features) for accuracy-focused strategies** (more correct predictions, better for sizing by confidence).

**Use Model B (55 features, cleaned) for PnL-focused strategies** (higher win rate on bets placed, better capital efficiency).

### Next steps

1. **Deeper reversal features** — use 20-30 candle lookback for regime detection (are we in a choppy or trending market?)
2. **Collect more data** — 869 training candles is the bottleneck. With 5,000+, the reversal indicators may start separating
3. **Two-stage model** — first predict "is this candle likely to reverse?" (binary), then predict direction only for non-reversal candles
4. **Try correlation threshold 0.90** instead of 0.85 — drops only 3 features, may preserve more signal while removing the worst redundancy

In [ ]:
print("=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)

acc_a = res_a["correct"] / res_a["total"] * 100
acc_b = res_b["correct"] / res_b["total"] * 100
rev_a = res_a["rev_correct"] / res_a["rev_total"] * 100
rev_b = res_b["rev_correct"] / res_b["rev_total"] * 100
nonrev_a = res_a["nonrev_correct"] / res_a["nonrev_total"] * 100
nonrev_b = res_b["nonrev_correct"] / res_b["nonrev_total"] * 100

print(f"\n{'Metric':<25} {'Model A (61 feat)':>18} {'Model B (clean)':>18} {'Diff':>8}")
print("-" * 72)
print(f"{'Per-candle accuracy':<25} {acc_a:>17.1f}% {acc_b:>17.1f}% {acc_b - acc_a:>+7.1f}%")
print(f"{'Reversal accuracy':<25} {rev_a:>17.1f}% {rev_b:>17.1f}% {rev_b - rev_a:>+7.1f}%")
print(f"{'Non-reversal accuracy':<25} {nonrev_a:>17.1f}% {nonrev_b:>17.1f}% {nonrev_b - nonrev_a:>+7.1f}%")
print(f"{'Features':<25} {len(all_features):>18} {len(clean_features):>18}")
print(f"{'Corr pairs dropped':<25} {'0':>18} {len(to_drop):>18}")